# Phase C v2 — Explanation Faithfulness (rigorous build)
### *Trustworthiness Under Compression*

Upgrades over v1 (which had an explainer-noise floor too high to report):
- **NSAMPLES=1500** (was 128) — stabilises KernelSHAP so the M0-vs-itself floor rises to >0.9, letting compression drift stand clearly above noise.
- **N_PER_CLASS=80, BG_K=50** — tighter per-class estimates, richer background.
- **Fresh cache path** (`results/shap_v2/`) — recomputes cleanly; v1 arrays untouched.
- **Bootstrap 95% CIs** (B=1000) on every agreement claim, judged against the seed null band — the rigor that makes the finding unattackable.

Same locked §4.0 protocol: one model-agnostic explainer applied identically to all models; fixed seeded eval subset + background; true-class attribution; three-floor noise hierarchy (explainer < seed < compression).

> SHAP cell is slow at these settings (~1.5–3 h, once, cached). Everything after is fast.

## 0. Setup

In [1]:
!pip install -q shap
import os, gc, json, time, random, glob
import numpy as np, pandas as pd, tensorflow as tf, joblib, shap
from scipy.stats import spearmanr, kendalltau

def seed_everything(s):
    os.environ['PYTHONHASHSEED']=str(s); random.seed(s); np.random.seed(s); tf.random.set_seed(s)
seed_everything(0)

ROOT='/content/drive/MyDrive/Trustworthiness_IoT'
if not os.path.isdir(f'{ROOT}/data'):
    from google.colab import drive; drive.mount('/content/drive')
PATHS={k:f'{ROOT}/{k}' for k in ['data','splits','models','results','logs']}
PATHS['cache']=f'{ROOT}/data/cache'; PATHS['shap']=f'{ROOT}/results/shap_v2'
os.makedirs(PATHS['shap'],exist_ok=True)
CLASSES=['Benign','DDoS','DoS','Mirai','Recon','Spoofing','Web','BruteForce']
class_to_idx={c:i for i,c in enumerate(CLASSES)}; N_CLASSES=len(CLASSES)

# ---- knobs (raised for reportable rigor) ----
N_PER_CLASS=80
BG_K=50
NSAMPLES=1500
B_BOOT=1000
print('shap',shap.__version__,'| TF',tf.__version__)

Mounted at /content/drive
shap 0.52.0 | TF 2.20.0


In [2]:
df=pd.read_parquet(f"{PATHS['cache']}/ciciot_processed.parquet")
FEATURES=[c for c in df.columns if c!='y']
sp=np.load(f"{PATHS['splits']}/split_primary.npz"); tr,te=sp['train'],sp['test']
scaler=joblib.load(f"{PATHS['models']}/scaler_primary.joblib")
Xall=df[FEATURES].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan).fillna(0.0).astype('float32')
y=df['y'].map(class_to_idx).astype('int64').values; del df; gc.collect()
Xtr=scaler.transform(Xall.iloc[tr]).astype('float32'); ytr=y[tr]
Xte=scaler.transform(Xall.iloc[te]).astype('float32'); yte=y[te]
del Xall; gc.collect()
N_FEATURES=Xtr.shape[1]; print('features',N_FEATURES)

features 46


## 1. §4.0 protocol — eval subset, background, unified predict

In [3]:
rng=np.random.RandomState(0); idx_sel=[]
for ci in range(N_CLASSES):
    pool=np.where(yte==ci)[0]; take=min(N_PER_CLASS,len(pool))
    idx_sel.append(rng.choice(pool,size=take,replace=False))
idx_sel=np.concatenate(idx_sel)
Xeval=Xte[idx_sel]; yeval=yte[idx_sel]
print('eval',Xeval.shape,'| per-class',[int((yeval==i).sum()) for i in range(N_CLASSES)])
bg=shap.kmeans(Xtr[rng.choice(len(Xtr),5000,replace=False)], BG_K)

def make_predict(path, kind):
    if kind=='keras':
        m=tf.keras.models.load_model(path)
        return lambda Z: m.predict(np.asarray(Z,'float32'),batch_size=4096,verbose=0)
    it=tf.lite.Interpreter(model_path=path)
    inp=it.get_input_details()[0]['index']; outd=it.get_output_details()[0]['index']
    def f(Z):
        Z=np.asarray(Z,'float32'); it.resize_tensor_input(inp,Z.shape); it.allocate_tensors()
        dt=it.get_input_details()[0]['dtype']
        it.set_tensor(inp,Z.astype(dt)); it.invoke()
        P=it.get_tensor(outd).astype('float32'); P=np.clip(P,0,None)
        s=P.sum(1,keepdims=True); s[s==0]=1.0; return P/s
    return f

MODELS={'M0_s0':(f"{PATHS['models']}/M0_seed0.keras",'keras'),
 'M0_s1':(f"{PATHS['models']}/M0_seed1.keras",'keras'),
 'M0_s2':(f"{PATHS['models']}/M0_seed2.keras",'keras'),
 'M2_prune50':(f"{PATHS['models']}/M2_prune50.keras",'keras'),
 'M2_prune80':(f"{PATHS['models']}/M2_prune80.keras",'keras'),
 'M3_distilled':(f"{PATHS['models']}/M3_distilled.keras",'keras'),
 'M1_int8':(f"{PATHS['models']}/M1_int8.tflite",'tflite'),
 'M1_float16':(f"{PATHS['models']}/M1_float16.tflite",'tflite')}
json.dump({'n_per_class':N_PER_CLASS,'bg_k':BG_K,'nsamples':NSAMPLES,'B_boot':B_BOOT,
  'eval_n':int(len(Xeval)),'explainer':'KernelSHAP','attribution_target':'true class','seed':0},
  open(f"{PATHS['logs']}/attribution_protocol_v2.json",'w'),indent=2)
print('protocol logged.')

eval (640, 46) | per-class [80, 80, 80, 80, 80, 80, 80, 80]
protocol logged.


## 2. SHAP (once, cached) — true-class attribution

In [4]:
def true_class_shap(name, path, kind, es=0):
    cache=f"{PATHS['shap']}/{name}_es{es}.npy"
    if os.path.exists(cache): print('cached',name,es); return np.load(cache)
    np.random.seed(es); f=make_predict(path,kind)
    sv=shap.KernelExplainer(f,bg).shap_values(Xeval,nsamples=NSAMPLES,silent=True)
    if isinstance(sv,list): sv=np.stack(sv,axis=-1)
    sv=np.asarray(sv)
    if sv.ndim==2: sv=sv[:,:,None]
    if sv.shape[-1]!=N_CLASSES and sv.shape[1]==N_CLASSES and sv.shape[2]==N_FEATURES:
        sv=np.transpose(sv,(0,2,1))
    assert sv.shape[0]==len(Xeval) and sv.shape[1]==N_FEATURES, f"bad shape {sv.shape}"
    nC=sv.shape[2]; A=np.stack([sv[i,:,min(yeval[i],nC-1)] for i in range(len(yeval))]).astype('float32')
    np.save(cache,A); print('computed',name,es,A.shape); return A

t0=time.time()
A_m0 =true_class_shap('M0_s0',*MODELS['M0_s0'],es=0)
A_m0b=true_class_shap('M0_s0',*MODELS['M0_s0'],es=7)
A_s1 =true_class_shap('M0_s1',*MODELS['M0_s1'],es=0)
A_s2 =true_class_shap('M0_s2',*MODELS['M0_s2'],es=0)
A_var={n:true_class_shap(n,*MODELS[n],es=0) for n in ['M2_prune50','M2_prune80','M3_distilled','M1_int8','M1_float16']}
print(f'SHAP done in {(time.time()-t0)/60:.1f} min')

computed M0_s0 0 (640, 46)
computed M0_s0 7 (640, 46)
computed M0_s1 0 (640, 46)
computed M0_s2 0 (640, 46)
computed M2_prune50 0 (640, 46)
computed M2_prune80 0 (640, 46)
computed M3_distilled 0 (640, 46)


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


computed M1_int8 0 (640, 46)
computed M1_float16 0 (640, 46)
SHAP done in 23.4 min


## 3. Agreement vs M0 — with bootstrap 95% CIs, vs null band + noise floor

In [5]:
def per_sample_spearman(A_ref,A_other):
    out=np.full(len(A_ref),np.nan)
    for i in range(len(A_ref)):
        a,b=A_ref[i],A_other[i]
        if np.std(a)>1e-12 and np.std(b)>1e-12:
            out[i]=spearmanr(a,b).correlation
    return out
def boot_ci(vals,B=B_BOOT):
    v=vals[~np.isnan(vals)]; r=np.random.RandomState(0)
    means=[v[r.randint(0,len(v),len(v))].mean() for _ in range(B)]
    return float(v.mean()), float(np.percentile(means,2.5)), float(np.percentile(means,97.5))

floor=per_sample_spearman(A_m0,A_m0b)
null=np.concatenate([per_sample_spearman(A_m0,A_s1),per_sample_spearman(A_m0,A_s2)])
fm,fl,fh=boot_ci(floor); nm,nl,nh=boot_ci(null)
print(f'EXPLAINER-NOISE FLOOR (M0 vs M0): {fm:.3f} [{fl:.3f},{fh:.3f}]')
print(f'SEED NULL BAND  (M0 vs seeds)  : {nm:.3f} [{nl:.3f},{nh:.3f}]')

rows=[{'comparison':'FLOOR_explainer','spearman':round(fm,3),'CI_low':round(fl,3),'CI_high':round(fh,3)},
      {'comparison':'NULL_seed','spearman':round(nm,3),'CI_low':round(nl,3),'CI_high':round(nh,3)}]
for n,A in A_var.items():
    m,l,h=boot_ci(per_sample_spearman(A_m0,A))
    below = 'YES' if h < nl else ('partial' if m<nm else 'no')
    rows.append({'comparison':n,'spearman':round(m,3),'CI_low':round(l,3),'CI_high':round(h,3),'below_null':below})
agt=pd.DataFrame(rows); pd.set_option('display.width',200)
print(agt.to_string(index=False)); agt.to_csv(f"{PATHS['results']}/faithfulness_agreement_v2.csv",index=False)

EXPLAINER-NOISE FLOOR (M0 vs M0): 0.773 [0.764,0.782]
SEED NULL BAND  (M0 vs seeds)  : 0.643 [0.635,0.651]
     comparison  spearman  CI_low  CI_high below_null
FLOOR_explainer     0.773   0.764    0.782        NaN
      NULL_seed     0.643   0.635    0.651        NaN
     M2_prune50     0.735   0.723    0.745         no
     M2_prune80     0.437   0.418    0.455        YES
   M3_distilled     0.524   0.510    0.538        YES
        M1_int8     0.847   0.833    0.861         no
     M1_float16     0.999   0.998    1.000         no


In [6]:
# Per-class Spearman vs M0, with bootstrap CI — the convergence evidence
def perclass(A_ref,A_other):
    out={}
    for ci,c in enumerate(CLASSES):
        v=per_sample_spearman(A_ref[yeval==ci],A_other[yeval==ci])
        m,l,h=boot_ci(v); out[c]=f"{m:.2f}[{l:.2f},{h:.2f}]"
    return out
pc=[{'model':n,**perclass(A_m0,A)} for n,A in A_var.items()]
pcdf=pd.DataFrame(pc); print(pcdf.to_string(index=False))
pcdf.to_csv(f"{PATHS['results']}/faithfulness_perclass_spearman_v2.csv",index=False)

       model          Benign            DDoS             DoS           Mirai           Recon        Spoofing             Web      BruteForce
  M2_prune50 0.78[0.76,0.81] 0.73[0.71,0.76] 0.80[0.78,0.82] 0.76[0.73,0.78] 0.77[0.73,0.80] 0.72[0.69,0.75] 0.71[0.68,0.74] 0.61[0.57,0.64]
  M2_prune80 0.52[0.48,0.56] 0.54[0.51,0.57] 0.59[0.56,0.61] 0.59[0.57,0.62] 0.42[0.37,0.46] 0.39[0.35,0.43] 0.22[0.18,0.27] 0.23[0.17,0.29]
M3_distilled 0.56[0.53,0.60] 0.63[0.61,0.66] 0.61[0.59,0.63] 0.48[0.46,0.51] 0.53[0.50,0.57] 0.46[0.41,0.51] 0.45[0.39,0.50] 0.47[0.42,0.52]
     M1_int8 0.84[0.81,0.87] 0.88[0.86,0.90] 0.62[0.54,0.70] 0.94[0.93,0.95] 0.89[0.87,0.90] 0.89[0.86,0.91] 0.90[0.88,0.92] 0.82[0.79,0.85]
  M1_float16 1.00[1.00,1.00] 1.00[1.00,1.00] 1.00[1.00,1.00] 1.00[1.00,1.00] 1.00[1.00,1.00] 1.00[1.00,1.00] 1.00[1.00,1.00] 1.00[0.99,1.00]


## 4. Perturbation faithfulness (AOPC, 3 maskers)

In [7]:
KS=[1,3,5,10]; RAND_DRAWS=20
mask_values={'mean':Xtr.mean(0),'zero':np.zeros(N_FEATURES,'float32')}
def mask_fill(masker,feats,seed):
    if masker in mask_values: return mask_values[masker][feats]
    return Xtr[np.random.RandomState(seed).randint(len(Xtr))][feats]
def comp(predict,X,A,masker):
    p0=predict(X); pred=p0.argmax(1); base=p0[np.arange(len(X)),pred]; out={}
    for k in KS:
        Xm=X.copy()
        for i in range(len(X)):
            tk=np.argsort(-np.abs(A[i]))[:k]; Xm[i,tk]=mask_fill(masker,tk,1000+i)
        out[k]=float(np.mean(base-predict(Xm)[np.arange(len(X)),pred]))
    aopc=np.mean(list(out.values())); randk=[]
    for k in KS:
        d=[]
        for j in range(RAND_DRAWS):
            r=np.random.RandomState(j); Xm=X.copy()
            for i in range(len(X)):
                rk=r.choice(N_FEATURES,k,replace=False); Xm[i,rk]=mask_fill(masker,rk,j*9999+i)
            d.append(np.mean(base-predict(Xm)[np.arange(len(X)),pred]))
        randk.append(np.mean(d))
    return aopc,np.mean(randk)
rows=[]
for name,(path,kind) in [('M0_s0',MODELS['M0_s0']),('M2_prune50',MODELS['M2_prune50']),
        ('M2_prune80',MODELS['M2_prune80']),('M3_distilled',MODELS['M3_distilled']),
        ('M1_int8',MODELS['M1_int8']),('M1_float16',MODELS['M1_float16'])]:
    A=A_m0 if name=='M0_s0' else A_var[name]; pred=make_predict(path,kind)
    for msk in ['mean','zero','marginal']:
        a,ar=comp(pred,Xeval.copy(),A,msk)
        rows.append({'model':name,'masker':msk,'AOPC_topk':round(a,4),'AOPC_randk':round(ar,4),'gap':round(a-ar,4)})
pf=pd.DataFrame(rows); print(pf.to_string(index=False)); pf.to_csv(f"{PATHS['results']}/faithfulness_perturbation_v2.csv",index=False)
print('\\nNote: on standardised features mean~=0, so mean & zero maskers coincide (expected).')

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


       model   masker  AOPC_topk  AOPC_randk    gap
       M0_s0     mean     0.5680      0.1565 0.4116
       M0_s0     zero     0.5680      0.1565 0.4116
       M0_s0 marginal     0.5532      0.1526 0.4006
  M2_prune50     mean     0.5984      0.1493 0.4491
  M2_prune50     zero     0.5984      0.1493 0.4491
  M2_prune50 marginal     0.5857      0.1510 0.4347
  M2_prune80     mean     0.5773      0.1034 0.4739
  M2_prune80     zero     0.5773      0.1034 0.4739
  M2_prune80 marginal     0.5304      0.1065 0.4239
M3_distilled     mean     0.6595      0.1838 0.4757
M3_distilled     zero     0.6595      0.1838 0.4757
M3_distilled marginal     0.6285      0.1887 0.4397
     M1_int8     mean     0.5217      0.1388 0.3829
     M1_int8     zero     0.5217      0.1388 0.3829
     M1_int8 marginal     0.5063      0.1370 0.3693
  M1_float16     mean     0.5678      0.1566 0.4113
  M1_float16     zero     0.5678      0.1566 0.4113
  M1_float16 marginal     0.5530      0.1526 0.4005
\nNote: on s

## 5. Manual commit

In [8]:
ROOT='/content/drive/MyDrive/Trustworthiness_IoT'
print('results:', [f for f in os.listdir(f'{ROOT}/results') if 'v2' in f])
# !git -C "{ROOT}" add notebooks/ results/ logs/
# !git -C "{ROOT}" commit -q -m "Phase C v2: stable SHAP (nsamples=1500), bootstrap-CI agreement, perturbation"
# !git -C "{ROOT}" push origin main

results: ['shap_v2', 'faithfulness_agreement_v2.csv', 'faithfulness_perclass_spearman_v2.csv', 'faithfulness_perturbation_v2.csv']


## Read the output
- **EXPLAINER-NOISE FLOOR** should now be HIGH (Spearman >0.9). If so, the noise problem is solved and drift is reportable.
- **Agreement table `below_null=YES`** means that variant's drift CI sits entirely below the seed null band — a clean, causal faithfulness loss. Expect M2_prune80 (and maybe M3) = YES; float16 = no (near-lossless).
- **Per-class CIs:** the convergence — do Web/BruteForce/Recon show the lowest agreement, matching their calibration collapse? Non-overlapping CIs vs the easy classes = the killer finding, bulletproofed.
- **Perturbation gap >0** across maskers = explanations behaviourally faithful; mean=zero expected.

**Next: Phase D** — TON_IoT transfer.